In [1]:
from netCDF4 import Dataset
import numpy as np
import pandas as pd
import netCDF4
from geopandas.tools import sjoin
from shapely.geometry import Point, shape, Polygon
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt

C:\Users\x12la\AppData\Local\Temp\ipykernel_23236\3396794205.py:5: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  from geopandas.tools import sjoin


In [2]:
# # Read 1.3 km grid and pass lat/lon to variable as arrays
grid = xr.open_dataset('C:/Users/x12la/Desktop/latlon_ChicagoLADCO_d03.nc')
cmaq_lon,cmaq_lat = np.array(grid['lon']),np.array(grid['lat'])
# laterial boundaries 
cmaq_llat,cmaq_ulat,cmaq_llon,cmaq_ulon=cmaq_lat.min(), cmaq_lat.max(), cmaq_lon.min(), cmaq_lon.max()

# corners from MCIP file -- doesn't matter dates
f = xr.open_dataset('../GRIDDOT2D_Chicago_LADCO_2018-08-16.nc')
lond,latd = np.array(f['LOND'][0][0]),np.array(f['LATD'][0][0])

# pull in lat lon data -- doesn't matter dates
d03 = xr.open_dataset('../GRIDCRO2D_Chicago_LADCO_2018-08-16.nc')  
lat3,lon3 = np.array(d03['LAT'][0][0]),np.array(d03['LON'][0][0])
lat,lon = lat3,lon3

In [3]:
# Import census track shapes to overlay emission data to 
tract_shapes = gpd.read_file('../demographic/US_tract_2018.shp')

In [5]:
def process_netcdf_to_shapefiles(nc_file, first_shapefile_name, second_shapefile_name):
    """
    Processes a NetCDF file to create shapefiles based on air quality data and census tracts.

    Parameters:
        nc_file (xarray.Dataset): The NetCDF file containing air quality data.
        first_shapefile_name (str): The name for the first saved shapefile.
        second_shapefile_name (str): The name for the second saved shapefile.

    Returns:
        None
    """

    import numpy as np
    import geopandas as gpd
    from shapely.geometry import Point, Polygon

    # ------------------------------------------------------------------
    # identify data variables with the expected CMAQ grid shape
    # ------------------------------------------------------------------
    datas_headers = []
    for v in nc_file.variables:
        if nc_file[v][0][0].shape == (288, 315):
            datas_headers.append(v)

    datas = []
    for i in range(len(datas_headers)):
        if nc_file[datas_headers[i]][0][0].shape == (288, 315):
            datas.append(np.array(nc_file[datas_headers[i]][0][0]).ravel())

    d = {}
    for i in range(len(datas)):
        d[datas_headers[i]] = datas[i]

    # ------------------------------------------------------------------
    # pull locations and put into shapely point so we can do
    # geographical transformations
    # ------------------------------------------------------------------
    pointline = [Point(lon.ravel()[i], lat.ravel()[i]) for i in range(len(lon.ravel()))]

    # add in geometry into dictionary
    d["geometry"] = pointline

    # make geodataframe using dictionary and keep lon/lat first
    gdf = gpd.GeoDataFrame(d, crs="EPSG:4326")

    # pull out these lat lon points
    x, y = gdf["geometry"].x, gdf["geometry"].y

    # ------------------------------------------------------------------
    # start to make little polygons to surround the centroids
    # ------------------------------------------------------------------
    shape = np.zeros(lon.shape).tolist()

    for i in range(len(lon)):
        for j in range(len(lon[0])):
            X0, Y0 = lat[i][j], lon[i][j]

            # get corners
            lo_lu, la_lu = latd[i][j], lond[i][j]
            lo_ru, la_ru = latd[i + 1][j], lond[i + 1][j]
            lo_rl, la_rl = latd[i + 1][j + 1], lond[i + 1][j + 1]
            lo_ll, la_ll = latd[i][j + 1], lond[i][j + 1]

            # create points of corners
            pointList = [
                Point(lo_lu, la_lu),
                Point(lo_ru, la_ru),
                Point(lo_rl, la_rl),
                Point(lo_ll, la_ll),
                Point(lo_lu, la_lu)
            ]

            # create a polygon from the corners
            poly = Polygon([[p.y, p.x] for p in pointList])

            # put the shape of the pixel into the shape list
            shape[i][j] = poly

    # ------------------------------------------------------------------
    # put list of shapes into final gdf, needs to be 1D
    # ------------------------------------------------------------------
    shape = np.array(shape).ravel()
    d["geometry"] = shape
    d["lat"], d["lon"] = lat.ravel(), lon.ravel()
    d["lat_m"], d["lon_m"] = x, y

    # make geodataframe of CMAQ grid polygons in geographic CRS
    gdf = gpd.GeoDataFrame(d, crs="EPSG:4326")
    gdf.to_file(first_shapefile_name)

    # ------------------------------------------------------------------
    # IMPORTANT: project both datasets to a projected CRS before
    # calculating intersection areas
    # ------------------------------------------------------------------
    # if tract_shapes is already projected, use that CRS
    # if tract_shapes is geographic, switch both to an equal-area/projected CRS
    if tract_shapes.crs is None:
        raise ValueError("tract_shapes must have a valid CRS before running this function.")

    if tract_shapes.crs.is_geographic:
        area_crs = "EPSG:5070"   # NAD83 / Conus Albers, good for area
        tract_shapes_proj = tract_shapes.to_crs(area_crs)
        gdf = gdf.to_crs(area_crs)
    else:
        area_crs = tract_shapes.crs
        tract_shapes_proj = tract_shapes.copy()
        gdf = gdf.to_crs(area_crs)

    gdf["geom_saved"] = gdf["geometry"]
    print("create polygon shapes")

    v = []
    geo = []
    gisjoin = []
    geoid = []

    # spatial join in projected CRS
    tractjoinfinal = gpd.sjoin(tract_shapes_proj, gdf, how="inner", predicate="intersects")
    grouped_tracts = tractjoinfinal.groupby("GISJOIN")

    print("joined tracts to polygon shapes")

    # ------------------------------------------------------------------
    # go through each join code and do computations to get final weighted AQ
    # ------------------------------------------------------------------
    for group in grouped_tracts.groups:
        # pull the section of the tract with repeating values
        tmp = grouped_tracts.get_group(group)

        # save geometry data from the census shape
        geo.append(tmp.iloc[0]["geometry"])

        # save the identifiers
        gisjoin.append(tmp.iloc[0]["GISJOIN"])
        geoid.append(tmp.iloc[0]["GEOID"])

        # get area of intersection of cells inside of the shape
        area = tmp.geometry.intersection(tmp.geom_saved).area

        # area-weight each air quality constituent
        for dh in datas_headers:
            v.append(np.sum(np.array(area) * tmp[dh]) / np.sum(np.array(area)))

    # ------------------------------------------------------------------
    # reshape and output
    # ------------------------------------------------------------------
    v = np.array(v)
    v2 = v.reshape(int(v.shape[0] / len(datas_headers)), len(datas_headers))
    v2 = gpd.GeoDataFrame(v2)
    v2.columns = datas_headers
    v2["GISJOIN"] = gisjoin
    v2["GEOID"] = geoid
    v2["geometry"] = geo
    v2.crs = tractjoinfinal.crs

    # optional: convert output back to tract_shapes original CRS
    if v2.crs != tract_shapes.crs:
        v2 = v2.to_crs(tract_shapes.crs)

    print("outputting to file")
    v2.to_file(second_shapefile_name)
    print("complete")

In [9]:
# EPA Default
COMBINE_ACONC_BASE_July2023_AnnualAvg = xr.open_dataset('COMBINE_ACONC_BASE_202301_D03_Jan2023_average.nc')
process_netcdf_to_shapefiles(COMBINE_ACONC_BASE_July2023_AnnualAvg, 'COMBINE_ACONC_BASE_Jan2023_average.shp', 'COMBINE_ACONC_BASE_Jan2023_average_censustracts.shp')

COMBINE_ACONC_BASE_July2023_AnnualAvg = xr.open_dataset('dailymaxozone_base_202301.nc')
process_netcdf_to_shapefiles(COMBINE_ACONC_BASE_July2023_AnnualAvg, 'MDA8O3_base_202301.shp', 'MDA8O3_base_202301_censustracts.shp')

# LOCUS Idling + HDV Mod 
COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg = xr.open_dataset('COMBINE_ACONC_LOCUSIdling_HDVSpatial_Jan2023_avg.nc')
process_netcdf_to_shapefiles(COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg, 'COMBINE_ACONC_LOCUSIdling_HDVSpatial_Jan2023_avg.shp', 'COMBINE_ACONC_LOCUSIdling_HDVSpatial_Jan2023_avg_censustracts.shp')

COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg = xr.open_dataset('dailymaxozone_LOCUSIdling_HDVSpatial_202301.nc')
process_netcdf_to_shapefiles(COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg, 'MDA8O3_LOCUSIdling_HDVSpatial_202301.shp', 'MDA8O3_LOCUSIdling_HDVSpatial_202301_censustracts.shp')

# # Renewal
COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg = xr.open_dataset('COMBINE_ACONC_Renewal_202301_D03_Jan2023_average.nc')
process_netcdf_to_shapefiles(COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg, 'COMBINE_ACONC_Renewal_Jan2023_average.shp', 'COMBINE_ACONC_Renewal_Jan2023_average_censustracts.shp')

COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg = xr.open_dataset('dailymaxozone_Renewal_202301.nc')
process_netcdf_to_shapefiles(COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg, 'MDA8O3_Renewal_202301.shp', 'MDA8O3_Renewal_202301_censustracts.shp')

# Zero-Pre2010s
COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg = xr.open_dataset('COMBINE_ACONC_ZeroPre2010s_Jan2023_avg.nc')
process_netcdf_to_shapefiles(COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg, 'COMBINE_ACONC_ZeroPre2010s_Jan2023_avg.shp', 'COMBINE_ACONC_ZeroPre2010s_Jan2023_avg_censustracts.shp')

COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg = xr.open_dataset('dailymaxozone_ZeroPre2010s_202301.nc')
process_netcdf_to_shapefiles(COMBINE_ACONC_LOCUSIdling_HDVSpatial_July2023_AnnualAvg, 'MDA8O3_ZeroPre2010s_202301.shp', 'MDA8O3_ZeroPre2010s_202301_censustracts.shp')

create polygon shapes
joined tracts to polygon shapes
outputting to file
complete
